In [1]:
from algorithms import (
    breadth_first_search_version_1, breadth_first_search_version_2,
    depth_first_search_version_1, depth_first_search_version_2,
    iterative_deepening_search_version_1, iterative_deepening_search_version_2,
    uniform_cost_search,
    greedy_search,
    a_star_search
)

In [2]:
import random

def gen_random_start_state(row_number=5, col_number=4):
    
    state = [[0] * col_number for _ in range(row_number)]

    # Vị trí robot: ô [0][0]
    state[0][0] = 'X'
 
    # 10% vật cản, 30% ô có rác, 60% ô trống
    for x in range(row_number):
        for y in range(col_number):
            if state[x][y] == 'X':
                continue

            r = random.random()
            if r < 0.1:
                state[x][y] = 2   
            elif r < 0.4:
                state[x][y] = 1  
 
    goal = [[0 if (cell == 'X' or cell == 1) else cell for cell in row] for row in state]
 
    return state, goal

In [3]:
import tkinter as tk
from tkinter import ttk
import time

ALGORITHM = {
    "BFS version 1": breadth_first_search_version_1,
    "BFS version 2": breadth_first_search_version_2,
    "DFS version 1": depth_first_search_version_1,
    "DFS version 2": depth_first_search_version_2,
    "IDS version 1": iterative_deepening_search_version_1,
    "IDS version 2": iterative_deepening_search_version_2,
    "UCS": uniform_cost_search,
    "Greedy search": greedy_search,
    "A*": a_star_search
}
 
class VacuumGUI:

    def __init__(self, window: tk.Tk):
        self.window = window
        self.window.title("Máy hút bụi")
        self.window.geometry("1280x720+120+50")
        self.window.configure(bg="#EEF5FB")

        self.start =  [['X', 0 , 1 , 1],
                       [ 0 , 2 , 0 , 0],
                       [ 1 , 0 , 1 , 2],
                       [ 2 , 1 , 1 , 1],
                       [ 1 , 0 , 1 , 1]]

        self.goal = [[0, 0, 0, 0],
                     [0, 2, 0, 0],
                     [0, 0, 0, 2],
                     [2, 0, 0, 0],
                     [0, 0, 0, 0]]

        self.animating = False
 
        self.build_layout()
        self.create_grid()
        self.draw_state(self.start)
 

    def build_layout(self):
        
        # Left - Phần setting
        LEFT_FRAME_BG = "#E8F1FD"
        
        self.left_frame = tk.Frame(self.window, bg=LEFT_FRAME_BG, width=200, padx=4, pady=10)
        self.left_frame.pack(side="left", fill="y", padx=8, pady=8)
        self.left_frame.pack_propagate(False)
 
        tk.Label(self.left_frame, text="CÀI ĐẶT", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 20, "bold")).pack(anchor="center")
 
        # Chọn thuật toán 
        tk.Label(self.left_frame, text="Thuật toán", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 12, "bold")).pack(anchor="w")
 
        self.algorithm_var = tk.StringVar() 
        self.algorithm_box = ttk.Combobox(self.left_frame, 
                                          textvariable=self.algorithm_var, 
                                          values=list(ALGORITHM.keys()), 
                                          state="readonly", width=25)

        self.algorithm_box.current(0) 
        self.algorithm_box.pack(pady=4) 
 
        tk.Button(self.left_frame, text="BẮT ĐẦU", command=self.start_algorithm, fg="white", bg="navy", width=15).pack(pady=5)
        tk.Button(self.left_frame, text="ĐẶT LẠI", command=self.reset, fg="white", bg="saddlebrown", width=15).pack(pady=5)
 
        tk.Frame(self.left_frame, bg="blue", height=1).pack(pady=10,fill="x")

        # Tạo input ngẫu nhiên
        tk.Label(self.left_frame, text="Tạo input ngẫu nhiên", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 12, "bold")).pack(anchor="w")

        resize_frame = tk.Frame(self.left_frame, bg=LEFT_FRAME_BG)
        resize_frame.pack(padx=10, pady=8,fill="x")
 
        tk.Label(resize_frame, text="Hàng:", fg="black").grid(row=0, column=0, sticky="w")
        self.row_var = tk.IntVar(value=5)
        tk.Spinbox(resize_frame, from_=2, to=7, textvariable=self.row_var, width=4, fg="black").grid(row=0, column=1, padx=4)
 
        tk.Label(resize_frame, text="Cột:", fg="black").grid(row=1, column=0, sticky="w")
        self.col_var = tk.IntVar(value=4)
        tk.Spinbox(resize_frame, from_=2, to=7, textvariable=self.col_var, width=4, fg="black").grid(row=1, column=1, padx=4)
 
        tk.Button(self.left_frame, text="TẠO", command=self.randomize_map, bg="darkgreen", fg="white", width=15).pack(pady=6)
 
        tk.Frame(self.left_frame, bg="blue", height=1).pack(pady=10,fill="x")

        # Chú thích
        tk.Label(self.left_frame, text="Chú thích", bg=LEFT_FRAME_BG, font=("Segoe UI", 12, "bold")).pack(anchor="w")
        
        tk.Label(self.left_frame, text="🤖: Robot", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=(8, 5), anchor="w")
        tk.Label(self.left_frame, text="🍂: Rác", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
        tk.Label(self.left_frame, text="🧱: Vật cản", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
        tk.Label(self.left_frame, text="Ô trống: Không có rác và vật cản", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
 

        # Center - phía trên: map, phía dưới: kết quả
        self.center_frame = tk.Frame(self.window, bg="#DCEEF8")
        self.center_frame.pack(side="left", fill="both", expand=True, padx=8, pady=8)
 
        self.map_frame = tk.LabelFrame(self.center_frame, text="Sơ đồ", 
                                       bg="#CFE7F5", fg="#0F172A",
                                       font=("Consolas", 13, "bold"),
                                       bd=2, labelanchor="n")
        self.map_frame.pack(fill="both", expand=True, padx=8, pady=4)
 
        self.result_frame = tk.LabelFrame(self.center_frame, text="Kết quả",
                                          bg="#CFE7F5", fg="#0F172A",
                                          font=("Consolas", 13, "bold"),
                                          bd=2, labelanchor="n")
        self.result_frame.pack(fill="x", padx=4, pady=4)
 
        self.result_text = tk.Text(self.result_frame, height=6,
                                   bg="white", fg="#2F4F5F",
                                   state="disabled")
        self.result_text.pack(fill="both", padx=4, pady=4)
 
        # Right - log
        self.log_frame = tk.LabelFrame(self.window, text="Log",
                                       bg="#E6F1F7", fg="#0E0C1B",
                                       font=("Consolas", 11, "bold"),
                                       bd=2, labelanchor="n")
        self.log_frame.pack(side="right", fill="y", padx=8, pady=8)
 
        self.log_scroll = tk.Scrollbar(self.log_frame)
        self.log_scroll.pack(side="right", fill="y")
 
        self.log_text = tk.Text(self.log_frame, width=33, 
                                bg="#E6F1F7", fg="#0F172A",
                                font=("Consolas", 9),
                                yscrollcommand=self.log_scroll.set,
                                state="disabled")
        self.log_text.pack(fill="both", expand=True, padx=4, pady=4)
        self.log_scroll.config(command=self.log_text.yview)
 

    def create_grid(self):
        for widget in self.map_frame.winfo_children():
            widget.destroy()
 
        self.cells = []
        row_number = len(self.start)
        col_number = len(self.start[0])
 
        self.map_frame.grid_columnconfigure(list(range(col_number+2)), weight=1)
        self.map_frame.grid_rowconfigure(list(range(row_number+2)), weight=1)
 
        for i in range(row_number):
            row = []
            for j in range(col_number):
                cell = tk.Label(self.map_frame,
                                width=4, height=2,
                                font=("Segoe UI Emoji", 18))
                cell.grid(row=i+1, column=j+1, padx=3, pady=3, sticky="nsew")
                row.append(cell)
            self.cells.append(row)
 

    def draw_state(self, state):
        cell_dict = {
            "X": {"icon" : "🤖", "color" : "#C7E4F4"},
            0: {"icon" : "", "color" : "#EAF4FA"},
            1: {"icon" : "🍂", "color" : "#DCEAD7"},
            2: {"icon" : "🧱", "color" : "#C8D3DC"}
        }

        for i in range(len(state)):
            for j in range(len(state[0])):
                v = state[i][j]
                self.cells[i][j].config(text=cell_dict.get(v).get("icon"), bg=cell_dict.get(v).get("color"))


    def animate_path(self, path):
        idx = 0
        for node in path:
            if not self.animating:
                return

            self.draw_state(node.state)
            step_label = "Khởi tạo" if node.action is None else node.action
            self.write_log(f"Bước: {idx}\nCost = {node.path_cost}\n{step_label}")
            for row in node.state:
                self.write_log("\t" + " ".join(map(str, row)))
            self.window.update()    
            time.sleep(0.6)

            idx += 1

        self.write_log("Phòng đã được dọn sạch!!!")
        self.animating = False


    def get_result(self, path):
        s = f"Số bước: {len(path)-1}\nChuỗi hành động: "
        for node in path:
            step_action = "Khởi tạo" if node.action is None else node.action
            s += step_action + " → "
        
        s += "Goal"
        return s
        

    def start_algorithm(self):
        if self.animating:
            return
 
        algo_name = self.algorithm_var.get()
        run_algo = ALGORITHM.get(algo_name)
        if run_algo is None:
            return
 
        self.clear_texts()
        self.draw_state(self.start)
 
        self.write_log(f"Thuật toán: {algo_name}")
        self.write_log("Đang tìm đường đi...\n")

        start_time = time.perf_counter()

        path = run_algo(self.start, self.goal)

        end_time = time.perf_counter()
        execution_time = end_time - start_time
 
        if path is None:
            self.write_log("Không tìm thấy đường đi!!!")
            self.write_result("Không tìm thấy đường đi!!!")
            return
        
        self.write_result(f"Thời gian tìm kiếm: {execution_time:.5f} s\n" + self.get_result(path))
        self.write_log("Đã tìm thấy cách di chuyển. \nBắt đầu hành động...\n")
        self.animating = True
        self.animate_path(path)
 

    def reset(self):
        self.animating = False
        self.draw_state(self.start)
        self.clear_texts()
 

    def randomize_map(self):
        self.animating = False
 
        row_number = self.row_var.get()
        col_number = self.col_var.get()
        self.start, self.goal = gen_random_start_state(row_number, col_number)
 
        self.create_grid()
        self.draw_state(self.start)
        self.clear_texts()


    def write_log(self, msg):
        self.log_text.config(state="normal")
        self.log_text.insert(tk.END, msg + "\n")
        self.log_text.see(tk.END)
        self.log_text.config(state="disabled")
 

    def write_result(self, msg):
        self.result_text.config(state="normal")
        self.result_text.insert(tk.END, msg)
        self.result_text.see(tk.END)
        self.result_text.config(state="disabled")
    

    def clear_texts(self):
        self.log_text.config(state="normal")
        self.result_text.config(state="normal")
        self.log_text.delete("1.0", tk.END)
        self.result_text.delete("1.0", tk.END)
        self.log_text.config(state="disabled")
        self.result_text.config(state="disabled")
        

if __name__ == "__main__":
    window = tk.Tk()
    app = VacuumGUI(window)
    window.mainloop()